<a href="https://colab.research.google.com/github/limaraujo/-Graduation-in-Computer-Science---CIN-UFPE/blob/main/%5BTrailKeeper%5D_Aula_pra%CC%81tica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TrailKeeper — Step 1: Setup & sanity checks

In this step we will:

* Install dependencies (into the current Jupyter kernel)
* (Optionally) log in to Weights & Biases
* Verify configs and scripts exist

In [1]:
REPO_URL = "https://github.com/MauricioSight/TrailKeeper.git"
REPO_DIR = "TrailKeeper"

import pathlib

if not pathlib.Path(REPO_DIR).exists():
    print("Cloning TrailKeeper...")
    !git clone $REPO_URL

# enter repo
%cd $REPO_DIR
print("Python version in this kernel:")
!python -V

Cloning TrailKeeper...
Cloning into 'TrailKeeper'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 119 (delta 37), reused 109 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 176.82 KiB | 2.81 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/TrailKeeper
Python version in this kernel:
Python 3.12.13


In [2]:
print("Installing dependencies (this uses the current Jupyter kernel)...")
!pip install -r requirements.txt

Installing dependencies (this uses the current Jupyter kernel)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.7 MB/s eta 0:00:00


### (Optional) Weights & Biases login

If you want experiment tracking dashboards, log in to W&B. You can either:

* run `!wandb login` and paste your API key, **or**
* set the environment variable `WANDB_API_KEY` beforehand.

In [ ]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 

In [ ]:
import os

if os.environ.get("WANDB_API_KEY"):
    print("WANDB_API_KEY detected. You're good to go")
else:
    print("No WANDB_API_KEY found.")
    print("If you want tracking, run this and follow the prompt:")
    print("    !wandb login")

### Quick sanity checks (configs & scripts)

In [ ]:
import glob, os, json, textwrap, pathlib

print("Configs available:")
for p in sorted(glob.glob("configs/*.yaml")):
    print(" -", p)

print("\nTop-level scripts:")
for p in sorted(glob.glob("execute_*.py")):
    print(" -", p)

# Show the MLP config
print("\nPreview configs/mlp.yaml:")
!sed -n '1,120p' configs/mlp.yaml

# Step 2: First Experiment

Now let’s run our **first training + validation** with the provided MLP config on MNIST.

What you’ll see:

* A new folder in `runs/` with timestamped name
* Saved **config**, **metrics**, **logs**, **predictions**, and **model weights**
* (Optional) live tracking in [W&B](https://wandb.ai/) if enabled

In [ ]:
!python execute_train_validation.py --config mlp

Every experiment is stored in the `runs/` folder.
Let’s peek at what was just created.

In [ ]:
import os

runs = sorted(os.listdir("runs"))
print("Available runs:", runs[-3:])  # last few runs
last_run = os.path.join("runs", runs[-1])
print("Latest run folder:", last_run)

print("\nFiles inside the run:")
os.listdir(last_run)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Look inside artifacts

* `config.yaml`: snapshot of hyperparameters
* `train_metrics.json` or `.csv`: evaluation results
* `train_output.*`: model raw outputs
* `model.pt` (or similar): saved weights

Let’s preview the metrics file.

In [ ]:
import json, glob

metrics_files = glob.glob(os.path.join(last_run, "train_metrics*"))
print("Metrics files:", metrics_files)

if metrics_files:
    with open(metrics_files[0]) as f:
        print(json.dumps(json.load(f), indent=2))

# Step 3: recompute metrics (no retraining)

every run saves **predictions** + **true labels**, so you can recompute metrics later on.
this is essential for:

* reproducibility
* trying new evaluation criteria
* fair comparisons across experiments

In [ ]:
last_run = runs[-1]

In [ ]:
!python execute_get_metrics.py --run_id $last_run

# Step 4: Manual hyperparameter changes

You can **manually edit the config** to try different hyperparameters.

open `configs/mlp.yaml` and experiment with values like:

```yaml
modeling:
  structure:
    num_layers: 16 # 👈 try fewer or more layers

  training:
    learning_rate: 0.0001  # 👈 try changing the lr
```

In [ ]:
!python execute_train_validation.py --config mlp

### compare results

1. check the new run folder in `runs/`
2. compare metrics with your previous run (accuracy, loss, etc.)
3. ask: *how did changing `lr` or `epochs` affect the outcome?*

In [ ]:
import os, json, glob

runs = sorted(os.listdir("runs"))
last_run = os.path.join("runs", runs[-1])
metrics_files = glob.glob(os.path.join(last_run, "train_metrics*"))

if metrics_files:
    with open(metrics_files[0]) as f:
        metrics = json.load(f)
    print("Metrics from last run:")
    print(json.dumps(metrics, indent=2))

# step 5: hyperparameter tuning with optuna

trailkeeper integrates with **optuna** to automate hyperparameter search.
you define a **search space** in a config file (e.g., `configs/tune-mlp.yaml`), and trailkeeper will:

* sample hyperparameters
* train models
* log metrics
* record the best trial

In [ ]:
!python execute_tunning.py --config tune-mlp

### inspect results

after tuning:

* check the `runs/` folder for multiple trial runs
* see the best parameters and score in the logs (and in W&B if enabled)

In [ ]:
import os

runs = sorted(os.listdir("runs"))
print("Last few runs after tuning:", runs[-5:])

# Step 6: Add a CNN model

In this step, we will integrate a **Convolutional Neural Network (CNN)** into TrailKeeper, using the changes from the PR. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)

* Inspect and understand the CNN implementation
* Register it in the model factory
* Add a config file for CNN
* Run training + evaluation with CNN
* Compare results with the MLP baseline

#### What changed / where to hook in

From the PR:

* **`configs/cnn.yaml`** was added, defining structure (input dimensions, kernel size, number of layers, channels, dropout, hidden size, output size) and training hyperparameters. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)
* **`modeling/structure/cnn.py`** contains the new `CNN` class extending `PytorchModelStructure`. It constructs the convolutional layers, pooling, flatten, linear layers, etc. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)
* **`modeling/structure/factory.py`** was updated: when `name == 'cnn'`, it now returns `CNN`. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)

So to integrate the CNN, you need:

1. The config file in `configs/` for `"cnn"`.
2. CNN class in `modeling/structure/`.
3. Factory mapping from `"cnn"` to the class.
4. (Optionally) In the YAML, reference preprocessing, metrics, etc., consistent with other models.

In [ ]:
!python execute_train_validation.py --config configs/cnn.yaml

### Inspect the CNN run & compare

* Look at the generated `runs/` folder.
* Open `metrics.json` / CSV of that run and compare with your MLP runs.
* Think: how does CNN perform compared to MLP? Where does it help (e.g. on image data)?

In [ ]:
import os, glob, json

runs = sorted(os.listdir("runs"))
latest = os.path.join("runs", runs[-1])
print("Latest run:", latest)

mf = glob.glob(os.path.join(latest, "metrics*"))
if mf:
    with open(mf[0]) as f:
        print("Metrics:", json.dumps(json.load(f), indent=2))

# Final step (exercise): add your own model

1. Make changes to the CNN and check if it achieves better results.
2. Find the best hyperparameters for both the CNN and the MLP models.
3. Analyze which model performs best and try to reason why.